# Pilot study on Colab A100 — self-repair SBN parsing

Colab port of the HPC pipeline in `colloquium_prep/pilot_eval/`. Runs all three
stages on one A100 session, resilient to disconnects (checkpoints to Drive).

1. **Inference** — LoRA Gemma-2-9B parses 836 sentences × 7 conditions (5852 rows).
2. **Scoring** — smatch++ F1 vs gold `mr`.
3. **Analysis** — summary tables + figures.

**Design:** clone the public repo and *import* the existing functions (no rewrite)
so the prompt stays byte-identical to training. Only Colab glue is new: Drive,
HF login, HF base-model download, and a checkpoint/resume wrapper.

**Before you run:** Runtime → Change runtime type → **A100 GPU** (+ High-RAM).

## 1. Runtime check

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > A100."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}  |  VRAM: {vram:.1f} GB  |  bf16 supported: {torch.cuda.is_bf16_supported()}")
if "A100" not in name:
    print("\n[warn] Not an A100. bf16 gemma-2-9b (~18GB) may not fit; enable USE_4BIT in the config cell.")
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)

## 2. Install dependencies
Colab ships torch. `mip`/PuLP are pulled in transitively by smatch++ for the ILP solver.

In [ ]:
%pip install -q -U transformers peft accelerate datasets smatchpp penman networkx pandas pyarrow

## 3. Clone repo & put `pilot_eval` + `sbn_lib` on the path
Imports `run_inference.py` and `evaluate_smatchpp.py` as modules (their `main()` is
guarded, so nothing runs on import). `evaluate_smatchpp` self-registers `sbn_lib`
via its own `sys.path.insert`, so the bundled PMB modules resolve correctly.

In [ ]:
import os, sys
REPO = "/content/lct_master_project"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/hongxuzhou/lct_master_project.git {REPO}
PILOT = f"{REPO}/colloquium_prep/pilot_eval"
for p in (PILOT, f"{PILOT}/sbn_lib"):
    if p not in sys.path:
        sys.path.insert(0, p)

from run_inference import INSTRUCTION, CONDITION_COLS, melt_wide_to_long, build_prompt, generate_batch
from evaluate_smatchpp import sbn_to_penman, make_scorer, evaluate_frame, print_summary
print("imported OK — conditions:", list(CONDITION_COLS.values()))

## 4. Mount Google Drive & define output paths
All outputs (and the inference checkpoint) live on Drive so a disconnect never loses work.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
OUT_DIR = Path("/content/drive/MyDrive/pilot_eval_run")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(exist_ok=True)

PRED_CKPT = str(OUT_DIR / "preds_long.parquet")        # Stage 1 output (resumable)
SCORED    = str(OUT_DIR / "preds_long_scored.parquet")  # Stage 2 output
print("outputs ->", OUT_DIR)

## 5. Hugging Face login
`google/gemma-2-9b-it` is **gated**: accept its license on the HF model page once, then
paste a token with read access. (You are `Shrikes`, who trained the adapter — you have access.)

In [ ]:
from huggingface_hub import login
login()  # paste token, or set HF_TOKEN and call login(os.environ['HF_TOKEN'])

## 6. Config

In [ ]:
BASE          = "google/gemma-2-9b-it"
ADAPTER       = "Shrikes/sbn-gemma2-9b-lora-pmb"
DATASET       = "Shrikes/self_repair_parsing_pilot_data"
SPLIT         = "train"
BATCH         = 8
MAX_NEW       = 512
SMOKE_LIMIT   = 16          # rows for the smoke test
FLUSH_EVERY   = 5          # checkpoint to Drive every N batches
USE_4BIT      = False      # True only if NOT on A100 (e.g. L4/T4); needs bitsandbytes
SOLVER        = "ilp"      # "ilp" exact (recommended) | "hillclimber" fast fallback

## 7. Load data — melt WIDE→LONG + build gold lookup
One HF load serves both: `melt_wide_to_long` (reused) gives the inputs; the `mr`
column gives the per-id gold for scoring.

In [ ]:
from datasets import load_dataset
wide = load_dataset(DATASET, split=SPLIT).to_pandas()
long_df = melt_wide_to_long(wide)
gold_lookup = dict(zip(wide["id"], wide["mr"]))
print(f"long rows: {len(long_df)}  |  gold ids: {len(gold_lookup)}")
long_df.head()

## 8. Load model (base from HF + LoRA adapter)
Tokenizer comes **from the adapter** (training-time chat template) and uses
left-padding for batched generation. `attn_implementation="eager"` is required for
Gemma-2 logit-softcap correctness.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
eot = tokenizer.convert_tokens_to_ids("<end_of_turn>")
eos_ids = [tokenizer.eos_token_id]
if eot is not None and eot != tokenizer.unk_token_id:
    eos_ids.append(eot)

quant = None
if USE_4BIT:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.bfloat16,
                               bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE, dtype=torch.bfloat16, quantization_config=quant,
    attn_implementation="eager", device_map={"": 0})
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()
print("model ready  |  eos_ids:", eos_ids)

## 9. Smoke test (16 rows)
Eyeball SBN-shaped output (boxes, roles, `>n`/`<n` indices) before committing hours.

In [ ]:
smoke = long_df.head(SMOKE_LIMIT)
prompts = [build_prompt(tokenizer, s) for s in smoke["input_nl"]]
smoke_preds = generate_batch(model, tokenizer, prompts, MAX_NEW, eos_ids)
for cond, txt, pred in zip(smoke["condition"], smoke["input_nl"], smoke_preds):
    print(f"[{cond}] {txt}\n  -> {pred[:200]}\n")

## 10. Full inference — checkpoint + resume
Writes the running predictions parquet to Drive every `FLUSH_EVERY` batches. On a
rerun it reads the checkpoint, skips already-done `(id, condition)` pairs, and
continues. Safe to re-run after a disconnect.

In [ ]:
import pandas as pd
KEYS = ["id", "condition"]

prev = pd.read_parquet(PRED_CKPT) if os.path.exists(PRED_CKPT) else None
done = set(zip(prev["id"], prev["condition"])) if prev is not None else set()
todo = long_df[~long_df.set_index(KEYS).index.isin(done)].reset_index(drop=True)
print(f"{len(done)} done, {len(todo)} to go (of {len(long_df)})")

new_rows = []
def flush():
    if not new_rows:
        return
    cur = pd.DataFrame(new_rows)
    out = pd.concat([prev, cur], ignore_index=True) if prev is not None else cur
    out = out[["id", "condition", "input_nl", "pred_sbn"]]
    tmp = PRED_CKPT + ".tmp"
    out.to_parquet(tmp, index=False)
    os.replace(tmp, PRED_CKPT)  # avoid a half-written parquet if killed mid-flush

n = len(todo)
for bi, start in enumerate(range(0, n, BATCH)):
    batch = todo.iloc[start:start + BATCH]
    prompts = [build_prompt(tokenizer, s) for s in batch["input_nl"]]
    preds = generate_batch(model, tokenizer, prompts, MAX_NEW, eos_ids)
    for (_, r), p in zip(batch.iterrows(), preds):
        new_rows.append({"id": r["id"], "condition": r["condition"],
                         "input_nl": r["input_nl"], "pred_sbn": p})
    if bi % FLUSH_EVERY == 0:
        flush()
    print(f"  generated {min(start + BATCH, n):>5}/{n}", flush=True)

flush()
final = pd.read_parquet(PRED_CKPT)
assert len(final) == len(long_df), f"{len(final)} != {len(long_df)} — rerun to finish"
assert not final.duplicated(KEYS).any(), "duplicate (id, condition)"
print(f"[done] {len(final)} predictions -> {PRED_CKPT}")

## 11. Score with smatch++
Reuses `evaluate_frame` (gold→Penman cached per id) and `print_summary`.

In [ ]:
preds_df = pd.read_parquet(PRED_CKPT)
scorer = make_scorer(SOLVER)
scored = evaluate_frame(preds_df, gold_lookup=gold_lookup,
                        id_col="id", pred_col="pred_sbn", scorer=scorer)
scored.to_parquet(SCORED, index=False)
print_summary(scored, "condition")
print(f"\nsaved -> {SCORED}")

## 12. Analysis — summary tables + figures
Ported from `analysis.ipynb`. Reads the scored table; figures saved to Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)

CONDITION_ORDER = ["gold", "repair_head", "repair_mid", "repair_tail",
                   "repair_head_interrug", "repair_mid_interrug", "repair_tail_interrug"]
POSITION_ORDER = ["head", "mid", "tail"]
STATUS_ORDER = ["success", "ill_formed", "parse_error", "gold_error", "smatch_error"]

df = pd.read_parquet(SCORED)
df["f1"] = pd.to_numeric(df["f1"], errors="coerce")
df["condition"] = pd.Categorical(df["condition"], categories=CONDITION_ORDER, ordered=True)
def position(c):
    for p in POSITION_ORDER:
        if f"_{p}" in c: return p
    return "—"
df["position"] = df["condition"].astype(str).map(position)
df["has_interrug"] = df["condition"].astype(str).str.contains("interrug")
df["f1_pen"] = df["f1"].where(df["status"] == "success", 0.0)
print(f"rows: {len(df)} | conditions: {df['condition'].nunique()} | ids: {df['id'].nunique()}")
print(df["status"].value_counts().to_string())

In [ ]:
def summarize(g):
    n = len(g); ok = (g["status"] == "success").sum()
    sf = g.loc[g["status"] == "success", "f1"]
    row = {"N": n, "success_n": ok, "success_%": round(100*ok/n, 1) if n else 0,
           "F1_A_succ_mean": round(sf.mean(), 4) if len(sf) else np.nan,
           "F1_A_succ_median": round(sf.median(), 4) if len(sf) else np.nan,
           "F1_B_penalized_mean": round(g["f1_pen"].mean(), 4)}
    for s in STATUS_ORDER:
        row[s] = int((g["status"] == s).sum())
    return pd.Series(row)

summary = df.groupby("condition", observed=True).apply(summarize).reindex(CONDITION_ORDER)
summary.to_csv(FIG_DIR / "summary_table.csv")
summary

In [ ]:
colors = ["#555555"] + sns.color_palette("muted", 6)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(summary.index, summary["success_%"], color=colors)
ax.axhline(summary.loc["gold", "success_%"], ls="--", c="#555555", lw=1, label="gold baseline")
ax.set_ylabel("success %"); ax.set_title("Parse success rate by condition")
ax.set_xticklabels(summary.index, rotation=30, ha="right"); ax.legend()
for i, v in enumerate(summary["success_%"]):
    ax.text(i, v + 1, f"{v:.0f}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_success_rate.png", dpi=150); plt.show()

In [ ]:
succ = df[df["status"] == "success"]
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=succ, x="condition", y="f1", order=CONDITION_ORDER, cut=0, inner="quartile", ax=ax)
ax.set_title("F1 distribution among successfully-parsed rows (metric A)")
ax.set_xticklabels(CONDITION_ORDER, rotation=30, ha="right"); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_f1_violin.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(summary.index, summary["F1_B_penalized_mean"], color=colors)
ax.axhline(summary.loc["gold", "F1_B_penalized_mean"], ls="--", c="#555555", lw=1, label="gold baseline")
ax.set_ylabel("mean F1 (penalized)"); ax.set_ylim(0, 1.02)
ax.set_title("Penalized mean F1 by condition (failures = 0)")
ax.set_xticklabels(summary.index, rotation=30, ha="right"); ax.legend()
for i, v in enumerate(summary["F1_B_penalized_mean"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_f1_penalized.png", dpi=150); plt.show()

In [ ]:
fail_cols = [s for s in STATUS_ORDER if s != "success"]
fails = summary[fail_cols]
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(summary)); pal = sns.color_palette("Reds", len(fail_cols))
for col, c in zip(fail_cols, pal):
    ax.bar(summary.index, fails[col], bottom=bottom, label=col, color=c)
    bottom += fails[col].values
ax.set_ylabel("# rows"); ax.set_title("Failure-type breakdown by condition")
ax.set_xticklabels(summary.index, rotation=30, ha="right"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_failure_breakdown.png", dpi=150); plt.show()

In [ ]:
rep = df[df["position"].isin(POSITION_ORDER)].copy()
piv = rep.groupby(["position", "has_interrug"], observed=True)["f1_pen"].mean().unstack()
piv = piv.reindex(POSITION_ORDER)
piv.columns = ["plain" if c is False else "+interregnum" for c in piv.columns]
ax = piv.plot(kind="bar", figsize=(8, 5), rot=0)
ax.axhline(summary.loc["gold", "F1_B_penalized_mean"], ls="--", c="#555555", lw=1, label="gold baseline")
ax.set_ylabel("mean F1 (penalized)"); ax.set_xlabel("reparandum position")
ax.set_title("Penalized F1 by position × interregnum"); ax.set_ylim(0, 1.02); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_position_interreg.png", dpi=150); plt.show()
piv

In [ ]:
wide_f1 = df.pivot_table(index="id", columns="condition", values="f1_pen", observed=True)
deltas = wide_f1.drop(columns=["gold"]).sub(wide_f1["gold"], axis=0)
delta_summary = deltas.mean().reindex([c for c in CONDITION_ORDER if c != "gold"]).rename("mean ΔF1 vs gold")
print("Mean per-item drop from gold (penalized F1):")
print(delta_summary.to_string())
fig, ax = plt.subplots(figsize=(9, 5))
delta_summary.plot(kind="barh", ax=ax, color=sns.color_palette("muted", 6))
ax.axvline(0, c="#333", lw=1); ax.set_xlabel("mean ΔF1 (condition − gold)")
ax.set_title("Per-item degradation from gold")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_delta_vs_gold.png", dpi=150); plt.show()

## Done
Scored table and figures are on Drive under `pilot_eval_run/`. Pull
`preds_long_scored.parquet` back for the colloquium write-up.

- Report **both** F1 metrics; A−B gap = ill-formed output vs wrong-but-valid structure.
- `gold` success% < 100 is the base parser's error floor; read degradation relative to it.
- Interregnum effect = the two bars at each position in the position×interregnum chart.